# CHENTO_LIMIT_BID v2 — validation backtest

Backtest the long-only sleeve `strategies/sleeves/chento_limit_bid/` against
the full 2022-01 → 2026-05 history. **This notebook imports the same `math.py`
helpers the live sleeve uses**, so what the backtest validates is what the
sleeve will execute.

Tested gates:
1. Cooldown 24h
2. NY-overlap window (UTC 12-17)
3. Mon/Tue/Wed only
4. Active swing base (36h cluster + 4% expansion within 3d)
5. Price within 1.2% above base low
6. Confluence score ≥ 3 (basis≤-2bp, funding<0, OI flushed ≥1.5%, spot CVD > 0)
7. MTF bias gate: net ∈ {+1,+2} OR signature == '--+++'

Execution (v2 staged scale-out):
- Market entry at trigger bar's close
- Stop at `base_low * (1 - 0.020)`
- T1 close 33% at `entry + 1R`
- T2 close 50% of remaining at `entry + 3R`
- Runner trail 5% below high-water mark (armed after T1)
- TIF 21 days

Pass criteria: weighted R per signal ≥ +0.5 net of cost, n ≥ 15 signals.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'data' / 'databases' / 'prod.db').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('could not locate prod.db')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import sqlite3
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('dark_background')

from strategies.sleeves.chento_limit_bid import math as cli_math
from strategies.sleeves.chento_limit_bid import config as cli_cfg
DB = ROOT / 'data' / 'databases' / 'prod.db'
print(f'DB: {DB}')
print(f'config: window={cli_cfg.BASE_WINDOW_HOURS}h, '
      f'cluster_pct={cli_cfg.BASE_CLUSTER_PCT}, '
      f'conf_min={cli_cfg.CONF_SCORE_MIN}, '
      f'mtf_accept={cli_cfg.MTF_NET_ACCEPT}, '
      f'cooldown_min={cli_cfg.COOLDOWN_MIN}')

## Load + enrich 15m frame (full history)

In [ ]:
def _load_table(table, ts_col='timestamp', ts_unit='s'):
    con = sqlite3.connect(str(DB))
    df = pd.read_sql(f'SELECT * FROM {table} ORDER BY {ts_col}', con)
    con.close()
    df['ts'] = pd.to_datetime(df[ts_col], unit=ts_unit, utc=True)
    df = df.set_index('ts')
    return df[~df.index.duplicated(keep='last')]

spot15 = _load_table('cd_spot_15m')
fut15  = _load_table('cd_futures_15m')
oi_h   = _load_table('cd_open_interest')
fund_h = _load_table('cd_funding_rate')
start = oi_h.index.min()
spot15 = spot15.loc[start:]; fut15 = fut15.loc[start:]; fund_h = fund_h.loc[start:]

f = pd.DataFrame(index=spot15.index)
f['spot_o'] = spot15['open']; f['spot_h'] = spot15['high']
f['spot_l'] = spot15['low'];  f['spot_c'] = spot15['close']
f['spot_cvd'] = spot15['volume_buy'] - spot15['volume_sell']
f['fut_c'] = fut15['close'].reindex(f.index)
f['basis'] = f['fut_c'] - f['spot_c']
f['basis_bp'] = f['basis'] / f['spot_c'] * 10000.0
f['oi'] = oi_h['oi_close'].reindex(f.index).ffill(limit=4)
f['funding'] = fund_h['fr_close'].reindex(f.index).ffill(limit=32)
f = f.dropna(subset=['spot_c', 'fut_c', 'oi', 'funding']).copy()
print(f'{len(f):,} enriched 15m bars  ({f.index.min()} → {f.index.max()})')

## Build MTF bias series (resample btc_1m → M/W/D/H4/H1)

In [ ]:
con = sqlite3.connect(str(DB))
m1_full = pd.read_sql(
    'SELECT open_time, open, high, low, close, volume FROM btc_1m ORDER BY open_time',
    con)
con.close()
m1_full['ts'] = pd.to_datetime(m1_full['open_time'], unit='ms', utc=True)
m1_full = m1_full.set_index('ts').drop(columns='open_time')
m1_full = m1_full[~m1_full.index.duplicated(keep='last')]
m1_full.columns = ['o', 'h', 'l', 'c', 'v']

def _resample(df, rule):
    return df.resample(rule).agg(o=('o','first'), h=('h','max'),
                                  l=('l','min'),   c=('c','last'),
                                  v=('v','sum')).dropna()

rules = {'M':'1ME','W':'1W','D':'1D','H4':'4h','H1':'1h'}
mtf_bias_map = {}
for label, rule in rules.items():
    cfg = cli_cfg.MTF_DEFS[label]
    tf_df = _resample(m1_full, rule)
    mtf_bias_map[label] = cli_math.compute_tf_bias_series(
        tf_df, period=cfg['period'], slope=cfg['slope'])
    print(f'  {label}: {len(tf_df)} bars, period={cfg["period"]}, slope={cfg["slope"]}')

## Tick-by-tick simulation

Iterate every 15m bar. At each one, apply the same gate sequence the live
sleeve uses (cooldown → time/day → base → approach → score → MTF). When all
pass, record the signal and walk forward to its exit (SL / TP / TIF).

In [ ]:
signals = []
last_trigger = None
cooldown_td = pd.Timedelta(minutes=cli_cfg.COOLDOWN_MIN)
f_idx = f.index

for i in range(len(f_idx)):
    now_ts = f_idx[i]
    now_dt = now_ts.to_pydatetime()
    if last_trigger is not None and (now_ts - last_trigger) < cooldown_td: continue
    if not cli_math.passes_time_gate(now_dt): continue
    if i < cli_cfg.BASE_WINDOW_HOURS * 4 + cli_cfg.BASE_EXPANSION_DAYS * 24 * 4 + 12: continue
    sub = f.iloc[:i+1]
    base = cli_math.detect_active_base(sub, now_ts)
    if base is None: continue
    current_price = float(sub['spot_c'].iloc[-1])
    if not cli_math.is_approaching_base(current_price, base['base_low'],
                                          cli_cfg.BASE_APPROACH_BAND_PCT): continue
    window = f.loc[base['base_start_ts']: base['base_end_ts']]
    score = cli_math.score_base_window(window)
    if score['conf_score'] < cli_cfg.CONF_SCORE_MIN: continue
    sig, net = cli_math.mtf_signature_at(now_ts, mtf_bias_map)
    if not cli_math.passes_mtf_gate(sig, net): continue

    entry_price = current_price
    stop_price = base['base_low'] * (1 - cli_cfg.STOP_OFFSET_PCT)
    risk = entry_price - stop_price
    if risk <= 0: continue
    tif_end = now_ts + pd.Timedelta(days=cli_cfg.TIF_DAYS)
    forward = f.loc[now_ts:tif_end]

    # v2 staged-exit simulation via the tier state machine.
    # We track remaining_qty as a fraction (1.0 = full position, 0.0 = closed).
    # When T1 fires, remaining drops to 1 - T1_CLOSE_PCT.
    # When T2 fires (off the remaining post-T1), drops by T2_CLOSE_PCT of remaining.
    # The runner closes on trail / stop / TIF.
    state = {"t1_done": False, "t2_done": False, "trail_armed": False,
             "high_water": entry_price, "active_stop": stop_price}
    remaining = 1.0
    realized_R = 0.0       # cumulative R from partial closes (cost-deducted)
    mfe_R = 0.0
    outcome_final = 'tif'
    exit_ts_final = forward.index[-1]
    exit_price_final = float(forward['spot_c'].iloc[-1])
    cost_per_unit = (cli_cfg.COST_BP_RT + cli_cfg.SLIPPAGE_BP_RT) / 10000.0  # per closed slice on its notional

    for ts_, bar in forward.iterrows():
        if ts_ == now_ts: continue
        bar_h = float(bar['spot_h']); bar_l = float(bar['spot_l'])
        mfe_R = max(mfe_R, (bar_h - entry_price) / risk)
        result = cli_math.evaluate_tier_transitions(
            state, bar_high=bar_h, bar_low=bar_l,
            entry=entry_price, stop_initial=stop_price,
            t1_r=cli_cfg.T1_R, t2_r=cli_cfg.T2_R, trail_pct=cli_cfg.TRAIL_PCT)
        new_state = result['new_state']; actions = result['actions']
        for act in actions:
            if act['kind'] == 't1':
                slice_pct_of_orig = cli_cfg.T1_CLOSE_PCT
                r_on_slice = (act['price'] - entry_price) / risk
                # cost on slice's notional, expressed in R units (per slice fraction)
                cost_r_slice = cost_per_unit * (act['price'] / risk) * slice_pct_of_orig
                realized_R += slice_pct_of_orig * r_on_slice - cost_r_slice
                remaining -= slice_pct_of_orig
            elif act['kind'] == 't2':
                slice_pct_of_orig = cli_cfg.T2_CLOSE_PCT * remaining
                r_on_slice = (act['price'] - entry_price) / risk
                cost_r_slice = cost_per_unit * (act['price'] / risk) * slice_pct_of_orig
                realized_R += slice_pct_of_orig * r_on_slice - cost_r_slice
                remaining -= slice_pct_of_orig
            elif act['kind'] == 'stop_exit':
                # close runner at the stop / trail price
                r_on_slice = (act['price'] - entry_price) / risk
                cost_r_slice = cost_per_unit * (act['price'] / risk) * remaining
                realized_R += remaining * r_on_slice - cost_r_slice
                outcome_final = ('trail_exit'
                                 if new_state.get('trail_armed') and new_state['active_stop'] > stop_price
                                 else 'stop')
                exit_ts_final = ts_
                exit_price_final = act['price']
                remaining = 0.0
                break
        state = new_state
        if remaining <= 1e-9:
            break

    if remaining > 0:
        # TIF expired with leftover (didn't trail-stop / SL)
        r_on_runner = (exit_price_final - entry_price) / risk
        cost_r_slice = cost_per_unit * (exit_price_final / risk) * remaining
        realized_R += remaining * r_on_runner - cost_r_slice
        outcome_final = 'tif'

    hold_h = (exit_ts_final - now_ts).total_seconds() / 3600
    signals.append({
        'now_ts': now_ts, 'entry': entry_price, 'stop': stop_price,
        'base_low': base['base_low'],
        'conf_score': score['conf_score'],
        'basis_bp_mean': score['basis_bp_mean'],
        'mtf_sig': sig, 'mtf_net': net,
        'exit_ts': exit_ts_final, 'exit_price': exit_price_final,
        'outcome': outcome_final,
        'r_net': realized_R,
        'mfe_R': mfe_R,
        'hold_h': hold_h,
        't1_done': state['t1_done'], 't2_done': state['t2_done'],
    })
    last_trigger = now_ts

sig_df = pd.DataFrame(signals)
print(f'\n{len(sig_df)} signals over {f.index.min()} → {f.index.max()}')
print(f'  net mean R: {sig_df["r_net"].mean():+.3f}')
print(f'  win rate (net>0): {(sig_df["r_net"]>0).mean():.1%}')
print(f'  outcomes: {sig_df["outcome"].value_counts().to_dict()}')
print(f'  T1 hit rate: {sig_df["t1_done"].mean():.1%}')
print(f'  T2 hit rate: {sig_df["t2_done"].mean():.1%}')
print(f'  median MFE (R): {sig_df["mfe_R"].median():.2f}')
print(f'  median hold (hours): {sig_df["hold_h"].median():.1f}')


## Year breakdown

In [ ]:
if not sig_df.empty:
    sig_df['year'] = pd.to_datetime(sig_df['now_ts']).dt.year
    by_yr = sig_df.groupby('year').agg(
        n=('r_net','size'), mean_R_net=('r_net','mean'),
        median_R=('r_net','median'),
        t1_rate=('t1_done','mean'),
        sl_rate=('outcome', lambda s: (s=='sl').mean()),
        median_hold_h=('hold_h','median'),
    ).round(3)
    print(by_yr)
else:
    print('No signals — relax thresholds or check data')

## Visualize equity curve (per-signal R)

In [ ]:
if not sig_df.empty:
    sig_df = sig_df.sort_values('now_ts').reset_index(drop=True)
    sig_df['cum_R_net'] = sig_df['r_net'].cumsum()
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(sig_df['now_ts'], sig_df['cum_R_net'], color='cyan', lw=1.2)
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_title(f'CHENTO_LIMIT_BID v1 — cum R per signal (n={len(sig_df)}, '
                 f'net mean {sig_df["r_net"].mean():+.2f}R)')
    ax.set_ylabel('cumulative R'); ax.set_xlabel('signal time')
    plt.tight_layout(); plt.show()

    # MTF breakdown
    print('\n=== signals by MTF cell ===')
    sig_df['mtf_bucket'] = sig_df.apply(
        lambda r: '--+++' if r['mtf_sig']=='--+++' else f'net_{int(r["mtf_net"]):+d}', axis=1)
    by_cell = sig_df.groupby('mtf_bucket').agg(
        n=('r_net','size'), mean_R=('r_net','mean'),
        tp_rate=('outcome', lambda s: (s=='tp').mean())
    ).round(3).sort_values('n', ascending=False)
    print(by_cell.to_string())

## Pass / fail vs the v1 promotion criteria

- Pass: mean R per signal ≥ +0.5 net of cost AND n ≥ 15 signals
- Marginal: 0 < mean R < +0.5 net, n ≥ 15
- Fail: mean R ≤ 0 OR n < 15

If pass → wire into `spec_json.sleeves.CHENTO_LIMIT_BID` with weight 2.0, k=5.
If marginal → tighten gates (raise conf_score min, narrow MTF window) and re-run.
If fail → strategy doesn't survive the systematization; revisit the discretionary parts.

In [ ]:
n = len(sig_df)
if n == 0:
    verdict = 'FAIL: no signals'
else:
    mr = sig_df['r_net'].mean()
    if mr >= 0.5 and n >= 15:
        verdict = f'PASS: mean R {mr:+.2f}, n={n}'
    elif mr > 0 and n >= 15:
        verdict = f'MARGINAL: mean R {mr:+.2f}, n={n}'
    elif n < 15:
        verdict = f'UNDERPOWERED: n={n} (<15)'
    else:
        verdict = f'FAIL: mean R {mr:+.2f}'
print(f'\n=== {verdict} ===')